In [26]:
%run -i cache_teacher_v3.py

Device: cuda
Checking for existing models in memory...
Tokenizer found in memory!
Loading teacher model natively...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading pure_golden_records.json...
Total golden records: 52
Allocated 42 for training and 10 for testing.
Pre-computing teacher hidden states for TRAIN set...


Caching Teacher States: 100%|██████████| 42/42 [01:31<00:00,  2.18s/it]

✅ Saved 42 pairs to teacher_hidden_states_phase3.pt
✅ Saved test_set_ids.json
🔥 CRITICAL: Restart your Colab Kernel / Runtime NOW to completely free VRAM before running train_adapter_v3.py!


In [30]:
import gc
import torch

# Delete the models from global memory
if 'teacher_model' in globals():
    del globals()['teacher_model']
if 'model' in globals():
    del globals()['model']

# Force Python garbage collection
gc.collect()

# Force PyTorch to empty the CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
print("Teacher model deleted and GPU VRAM completely flushed!")

Teacher model deleted and GPU VRAM completely flushed!


In [29]:
import gc
import torch

# Delete the model with the old Rank 4 adapter glued to it
if 'student_model' in globals():
    del student_model

# Force PyTorch and Python to empty the GPU memory
gc.collect()
torch.cuda.empty_cache()

print("GPU Memory Cleared. Ready for Rank 32!")

GPU Memory Cleared. Ready for Rank 32!


In [31]:
import torch

# Load the newly generated cache file
data = torch.load("teacher_hidden_states_phase3.pt", weights_only=False)

print(f"Total training examples cached: {len(data)}\n")

# Let's inspect the first 3 examples to verify the token alignment
for i in range(15):
    s_inputs_cpu, t_hs_sparse, correct_idx, valid_ids = data[i]
    
    # t_hs_sparse is a dictionary mapping layer indices to hidden state tensors.
    # Grab the highest layer number key dynamically (e.g. 35) to avoid KeyErrors
    last_layer_key = list(t_hs_sparse.keys())[-1]
    last_layer_hs = t_hs_sparse[last_layer_key]
    
    # The sequence length (LCS) is the second dimension of the tensor
    lcs_len = last_layer_hs.shape[1]
    s_len = s_inputs_cpu['input_ids'].shape[1]
    
    print(f"Example {i+1}:")
    print(f"  - Student Total Tokens: {s_len}")
    print(f"  - Perfectly Aligned LCS Tokens: {lcs_len}")
    print(f"  - Saved Teacher Tensor Shape: {list(last_layer_hs.shape)}\n")

Total training examples cached: 42

Example 1:
  - Student Total Tokens: 90
  - Perfectly Aligned LCS Tokens: 66
  - Saved Teacher Tensor Shape: [1, 66, 2048]

Example 2:
  - Student Total Tokens: 90
  - Perfectly Aligned LCS Tokens: 66
  - Saved Teacher Tensor Shape: [1, 66, 2048]

Example 3:
  - Student Total Tokens: 163
  - Perfectly Aligned LCS Tokens: 139
  - Saved Teacher Tensor Shape: [1, 139, 2048]

Example 4:
  - Student Total Tokens: 68
  - Perfectly Aligned LCS Tokens: 44
  - Saved Teacher Tensor Shape: [1, 44, 2048]

Example 5:
  - Student Total Tokens: 84
  - Perfectly Aligned LCS Tokens: 60
  - Saved Teacher Tensor Shape: [1, 60, 2048]

Example 6:
  - Student Total Tokens: 43
  - Perfectly Aligned LCS Tokens: 19
  - Saved Teacher Tensor Shape: [1, 19, 2048]

Example 7:
  - Student Total Tokens: 74
  - Perfectly Aligned LCS Tokens: 50
  - Saved Teacher Tensor Shape: [1, 50, 2048]

Example 8:
  - Student Total Tokens: 43
  - Perfectly Aligned LCS Tokens: 19
  - Saved Teache

In [32]:
%run -i train_adapter_v3.py

Device: cuda
Checking for existing models in memory...
Tokenizer found in memory!
Loading student model natively...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded 52 golden records from pure_golden_records.json
Injected adapter into layer 32
Injected adapter into layer 33
Injected adapter into layer 34
Injected adapter into layer 35
Ultra-Sparse Injection complete. Total trainable parameters: 532,544
Attempting to load teacher_hidden_states_phase3.pt...
Loaded 42 pairs.
No checkpoint found — starting fresh from Epoch 1.

Training Epochs 1 → 10
Epoch 001 | Loss: 6509.9120  CE: 5.5578  Hyp: 6507.1331
Epoch 002 | Loss: 6505.4649  CE: 5.4883  Hyp: 6502.7208
Epoch 003 | Loss: 6501.7555  CE: 5.4045  Hyp: 6499.0533
Epoch 004 | Loss: 6498.2128  CE: 5.3254  Hyp: 6495.5501
Epoch 005 | Loss: 6494.6768  CE: 5.2054  Hyp: 6492.0741
Epoch 006 | Loss: 6491.0474  CE: 5.0851  Hyp: 6488.5049
Epoch 007 | Loss: 6487.4510  CE: 4.9010  Hyp: 6485.0005
Epoch 008 | Loss: 6484.1790  CE: 4.6561  Hyp: 6481.8509
Epoch 009 | Loss: 6481.0747  CE: 4.3240  Hyp: 6478.9127
Epoch 010 | Loss: 6478.0675  CE: 3.9564  Hyp: 6476.0893

Checkpoint saved → adapter_epoch_010.pt

--- 

In [33]:
%run -i train_adapter_v3.py

Device: cuda
Checking for existing models in memory...
Tokenizer found in memory!
Student model found in memory!
Loaded 52 golden records from pure_golden_records.json
Ultra-Sparse Injection complete. Total trainable parameters: 532,544
Attempting to load teacher_hidden_states_phase3.pt...
Loaded 42 pairs.
Resumed from adapter_epoch_010.pt — starting at Epoch 11.

Training Epochs 11 → 20
Epoch 011 | Loss: 6475.3003  CE: 3.5871  Hyp: 6473.5068
Epoch 012 | Loss: 6472.6765  CE: 3.2689  Hyp: 6471.0420
Epoch 013 | Loss: 6470.1284  CE: 2.9708  Hyp: 6468.6430
Epoch 014 | Loss: 6467.7991  CE: 2.7563  Hyp: 6466.4209
Epoch 015 | Loss: 6465.8185  CE: 2.5978  Hyp: 6464.5196
Epoch 016 | Loss: 6463.8858  CE: 2.4692  Hyp: 6462.6512
Epoch 017 | Loss: 6462.1332  CE: 2.3365  Hyp: 6460.9650
Epoch 018 | Loss: 6460.4498  CE: 2.2271  Hyp: 6459.3363
Epoch 019 | Loss: 6458.7409  CE: 2.1485  Hyp: 6457.6666
Epoch 020 | Loss: 6457.0346  CE: 2.0749  Hyp: 6455.9971

Checkpoint saved → adapter_epoch_020.pt

--- EVA

In [34]:
%run -i train_adapter_v3.py

Device: cuda
Checking for existing models in memory...
Tokenizer found in memory!
Student model found in memory!
Loaded 52 golden records from pure_golden_records.json
Ultra-Sparse Injection complete. Total trainable parameters: 532,544
Attempting to load teacher_hidden_states_phase3.pt...
Loaded 42 pairs.
Resumed from adapter_epoch_020.pt — starting at Epoch 21.

Training Epochs 21 → 30
Epoch 021 | Loss: 6455.4628  CE: 2.0299  Hyp: 6454.4478
Epoch 022 | Loss: 6453.9969  CE: 1.9747  Hyp: 6453.0096
Epoch 023 | Loss: 6452.5182  CE: 1.9225  Hyp: 6451.5570
Epoch 024 | Loss: 6451.0970  CE: 1.8810  Hyp: 6450.1565
Epoch 025 | Loss: 6449.7256  CE: 1.8489  Hyp: 6448.8011
Epoch 026 | Loss: 6448.2607  CE: 1.8137  Hyp: 6447.3538
Epoch 027 | Loss: 6446.8766  CE: 1.7747  Hyp: 6445.9893
Epoch 028 | Loss: 6445.6268  CE: 1.7460  Hyp: 6444.7539
Epoch 029 | Loss: 6444.5369  CE: 1.7107  Hyp: 6443.6816
Epoch 030 | Loss: 6443.4703  CE: 1.6783  Hyp: 6442.6311

Checkpoint saved → adapter_epoch_030.pt

--- EVA

In [16]:
import torch
import json
import torch.nn.functional as F

print("\n--- EVALUATION ---")
TASK_CONFIGS = {
    "boolean_expressions": ["True", "False"],
    "date_understanding": ["A", "B", "C", "D", "E", "F"],
    "logical_deduction_five_objects": ["A", "B", "C", "D", "E"],
    "navigate": ["Yes", "No"],
    "ruin_names": ["A", "B", "C", "D"],
    "snarks": ["A", "B"],
}

OPTION_TOKENS = {}
for task_name, options in TASK_CONFIGS.items():
    for opt in options:
        if opt not in OPTION_TOKENS:
            tok_ids = tokenizer.encode(" " + opt, add_special_tokens=False)
            OPTION_TOKENS[opt] = tok_ids[0]

def predict_mc(model, prompt, valid_options):
    msgs = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + "Answer:"
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1)
    raw = [probs[0, OPTION_TOKENS[opt]].item() for opt in valid_options]
    total = sum(raw)
    if total == 0: return valid_options[0], 0.0
    renorm = [r / total for r in raw]
    best = renorm.index(max(renorm))
    return valid_options[best], renorm[best]

# Load dataset
with open("final_golden_records.json", "r") as f:
    golden_records = json.load(f)

# Force fallback to the last 10 records safely
TEST_TASKS = golden_records[-10:] if len(golden_records) >= 10 else golden_records

print(f"Evaluating on {len(TEST_TASKS)} test records...\n")
student_model.eval()
correct = 0

print(f"{'ID':<6} {'Task':<30} {'Ans':>5} | {'Pred':>5}")
print("-" * 65)
for t in TEST_TASKS:
    valid_options = TASK_CONFIGS[t["task"]]
    pred, prob = predict_mc(student_model, t["student_prompt"], valid_options)
    is_correct = (pred == t["target"])
    correct += is_correct
    mark = "✅" if is_correct else "❌"
    print(f"{t.get('id', '?'):<6} {t['task'][:28]:<30} {t['target']:>5} | {pred:>5} {mark}")

print("-" * 65)
print(f"Accuracy: {correct}/{len(TEST_TASKS)} ({correct/len(TEST_TASKS):.0%})")


--- EVALUATION ---
Evaluating on 10 test records...

ID     Task                             Ans |  Pred
-----------------------------------------------------------------
223    boolean_expressions             True | False ❌
230    ruin_names                         B |     B ✅
234    boolean_expressions            False | False ✅
240    navigate                         Yes |    No ❌
242    date_understanding                 C |     D ❌
243    navigate                         Yes |    No ❌
244    ruin_names                         D |     D ✅
246    boolean_expressions             True | False ❌
247    ruin_names                         D |     D ✅
248    date_understanding                 E |     E ✅
-----------------------------------------------------------------
Accuracy: 5/10 (50%)


In [25]:
import json
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

# 1. Clear memory
if 'student_model' in globals():
    del student_model
gc.collect()
torch.cuda.empty_cache()

# 2. Load fresh base model
print("Loading fresh base model...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", 
    torch_dtype=torch.bfloat16, 
    attn_implementation="eager"
).to(device)
base_model.eval()

TASK_CONFIGS = {
    "boolean_expressions": ["True", "False"],
    "date_understanding": ["A", "B", "C", "D", "E", "F"],
    "logical_deduction_five_objects": ["A", "B", "C", "D", "E"],
    "navigate": ["Yes", "No"],
    "ruin_names": ["A", "B", "C", "D"],
    "snarks": ["A", "B"],
}

OPTION_TOKENS = {}
for task_name, options in TASK_CONFIGS.items():
    for opt in options:
        if opt not in OPTION_TOKENS:
            tok_ids = tokenizer.encode(" " + opt, add_special_tokens=False)
            OPTION_TOKENS[opt] = tok_ids[0]

def predict_mc(model, prompt, valid_options):
    msgs = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + "Answer:"
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1)
    raw = [probs[0, OPTION_TOKENS[opt]].item() for opt in valid_options]
    total = sum(raw)
    if total == 0: return valid_options[0]
    renorm = [r / total for r in raw]
    best = renorm.index(max(renorm))
    return valid_options[best]

# 3. Load all 90 records
with open("final_golden_records.json", "r") as f:
    golden_records = json.load(f)

print(f"\nPurging contaminated records from {len(golden_records)} total records...\n")
pure_golden_records = []

for t in golden_records:
    valid_options = TASK_CONFIGS[t["task"]]
    pred = predict_mc(base_model, t["student_prompt"], valid_options)
    
    # It is ONLY a true golden record if the properly formatted base model FAILS.
    if pred != t["target"]:
        pure_golden_records.append(t)

print("=" * 60)
print(f"Purge Complete!")
print(f"Original Records: {len(golden_records)}")
print(f"True Golden Records Remaining: {len(pure_golden_records)}")
print(f"Contaminated Records Removed: {len(golden_records) - len(pure_golden_records)}")
print("=" * 60)

# Save the mathematically pure dataset
with open("pure_golden_records.json", "w") as f:
    json.dump(pure_golden_records, f, indent=2)

print("Saved pure dataset to 'pure_golden_records.json'.")

Loading fresh base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Purging contaminated records from 90 total records...

Purge Complete!
Original Records: 90
True Golden Records Remaining: 52
Contaminated Records Removed: 38
Saved pure dataset to 'pure_golden_records.json'.
